# Explore the Mirrored Database

The Azure SQL database is being **mirrored** into Fabric: its tables are
replicated into OneLake as Delta tables, automatically and continuously —
no pipelines, no schedules, no code.

This notebook reads those replicated tables straight from OneLake and runs
cross-table analytics on them, side by side with the lakehouse.

In [ ]:
# The mirrored database replicates into OneLake — addressable like any item
WORKSPACE_ID = "{{WORKSPACE_ID}}"
MIRRORED_DB_ID = "{{MIRRORED_DB_ID}}"

def mirrored(table: str):
    path = (
        f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
        f"{MIRRORED_DB_ID}/Tables/dbo/{table}"
    )
    return spark.read.format("delta").load(path)

for t in ["products", "stores", "pos_transactions", "inventory_snapshots"]:
    print(f"dbo.{t}: {mirrored(t).count():,} rows replicated")

In [ ]:
# These are live SQL tables — query them like local Delta
products = mirrored("products")
stores = mirrored("stores")
txns = mirrored("pos_transactions")

products.createOrReplaceTempView("m_products")
stores.createOrReplaceTempView("m_stores")
txns.createOrReplaceTempView("m_txns")

print("Sample of mirrored pos_transactions:")
txns.orderBy("transaction_timestamp", ascending=False).show(5, truncate=False)

In [ ]:
# Cross-table analytics on replicated operational data — zero ETL happened
revenue = spark.sql("""
    SELECT p.category,
           s.region,
           COUNT(*)                                                   AS transactions,
           ROUND(SUM(t.quantity * t.unit_price * (1 - t.discount_pct/100)), 0) AS revenue
    FROM m_txns t
    JOIN m_products p ON t.product_id = p.sku
    JOIN m_stores s   ON t.store_id   = s.store_id
    GROUP BY p.category, s.region
    ORDER BY revenue DESC
""")
revenue.show(20, truncate=False)

In [ ]:
# Freshness: when did the newest source row land here?
from pyspark.sql.functions import max as spark_max

latest = txns.agg(spark_max("transaction_timestamp").alias("latest")).collect()[0]["latest"]
print(f"Most recent transaction visible in Fabric: {latest}")
print("\nEverything above is served from OneLake Delta files that Fabric")
print("keeps in sync with Azure SQL automatically. Open 02_live_change to")
print("watch a source-side change replicate in near-real time.")